# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
try:
    import duckdb
except ImportError:
    get_ipython().system('pip -q install duckdb huggingface_hub')
    import duckdb

import os

def _get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    from huggingface_hub import get_token
    return get_token()

HF_TOKEN = _get_hf_token()
assert HF_TOKEN, "No HF token found. Run `hf auth login` locally, or set an HF_TOKEN Colab Secret."

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month, per the assignment card -- never the _sample (last month, sealed)
MARCH = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"

print("Warehouse connection ready. Querying month =", MONTH)

# cheap sanity check only (no full scan) -- the actual verification queries,
# including row count and date span, are all in section 3 (exactly three,
# as the assignment asks for).
sample = con.execute(f"SELECT * FROM '{MARCH}' LIMIT 3").df()
print(f"Connected. Sample columns: {list(sample.columns)}")
print(f"Sample report_date values: {sorted(sample['report_date'].astype(str).tolist())}")

Warehouse connection ready. Querying month = 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Connected. Sample columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Sample report_date values: ['2026-03-01', '2026-03-01', '2026-03-01']


### 1. Unit of analysis + time window

**One row = one content item, on one day** (`client_hash_id` × `content_hash_id` × `report_date`)
— the natural grain of `fact_content_daily_performance`. **Time window: `month=2026-03`**, a
mid-panel month, not the `_sample` table — the assignment card is explicit that `_sample` is the
snapshot's LAST month (June 2026) and developing label logic there means peeking at the future
you'd be predicting. Connection and schema confirmed above; the grain and window claims are
verified with real queries in section 3 (queries 1 and 2 of 3).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# No query needed to classify fields -- but I do verify the columns I'm about
# to classify actually exist with these names, rather than guessing.
schema_fact = con.execute(f"DESCRIBE SELECT * FROM '{MARCH}'").df()["column_name"].tolist()
schema_content = con.execute(f"DESCRIBE SELECT * FROM '{BASE}/dim_content.parquet'").df()["column_name"].tolist()
print("fact_content_daily_performance columns:", schema_fact)
print("\ndim_content columns:", schema_content)

fact_content_daily_performance columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized

### 2. Fields: feature / label / context / excluded

**Context** (grouping/joining/reading only, never a feature): `client_hash_id`,
`content_hash_id`, `report_date`, `month`.

**Feature** (knowable as of `report_date`, no future peeking): `gsc_avg_position`,
`gsc_impressions`, `gsc_clicks` (→ CTR), `sessions_organic`, and — gated on
`ga4_data_available IS TRUE` — `ga4_engaged_sessions` / `ga4_sessions` (→ engagement rate). All
five come straight from that day's own row in the fact table; none require joining a dimension
table, for a reason explained below.

**Label / proxy**: not a stored column — I'd build it the same way the starter dataset's
`trend_direction` was built, but at warehouse scale: compare a content item's `gsc_impressions`
in one 30-day window against the prior 30-day window, threshold at -20% decline, exactly as
`docs/data-dictionary.md` documents for the starter CSV. It's observed (measured from real
daily search performance), not a rule I invented, and it only exists in a *later* window relative
to the features — which is the whole point of section 3's trap below.

**Excluded, each with a reason**:
- `dim_content.content_updated_date`, `last_optimized_date`, `optimization_eligible_date` —
  **not safe as-is**. Verified below: `dim_content` is a single current-state snapshot (as of
  the July 2026 export), not a point-in-time dimension, so joining its "last updated" date
  against a March fact row can put the update *after* the day being described. Using it naively
  as "days since update" would leak the future into a feature.
- `is_published`, `is_deleted` — product/editorial decision flags, not observed search or
  analytics behavior (mirrors the flyrank-data skill's rule against product-decision columns).
- `provider_used`, `model_used` — explicitly "not a model feature" per the data dictionary
  convention carried over from the starter dataset.
- `keyword_hash_id`, `url_hash_id` — identifiers, grouping/joining only.
- Any `month=2026-04` (or later) row, for any feature — that's the outcome window, not the
  decision moment.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# --- Query 1 of 3: grain -- one row really is one (report_date, client, content)? ---
grain_check = con.execute(f"""
  SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
  FROM '{MARCH}'
  GROUP BY 1, 2, 3
  HAVING COUNT(*) > 1
  LIMIT 5
""").df()
print("Duplicate (report_date, client, content) combos found:", len(grain_check), "(0 = grain holds)")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client, content) combos found: 0 (0 = grain holds)
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


**Query 1 of 3 — grain.** Zero duplicate `(report_date, client, content)` combinations came
back — the stated grain (one row = one content item, one day) holds.

In [4]:
# --- Query 2 of 3: row count and date span for this slice ---
counts = con.execute(f"""
  SELECT COUNT(*) n, MIN(report_date) d0, MAX(report_date) d1,
         COUNT(DISTINCT client_hash_id) clients, COUNT(DISTINCT content_hash_id) items
  FROM '{MARCH}'
""").df()
print(counts.to_string(index=False))

      n         d0         d1  clients  items
9841378 2026-03-01 2026-03-31       55 331437


**Query 2 of 3 — row count and date span.** The March partition holds 9,841,378 rows spanning
2026-03-01 through 2026-03-31 (all 31 days present), across 55 distinct clients and 331,437
distinct content items — confirming both the stated time window and that this slice is a real,
usable size to build features and a proxy label on without touching the sealed final month.

In [5]:
# --- Query 3 of 3: availability, filtered with `IS TRUE` on purpose --
# ga4_data_available is THREE-valued (TRUE / FALSE / NULL), not two.
availability = con.execute(f"""
  SELECT
    SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS is_true,
    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS is_false,
    SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS is_null,
    COUNT(*) AS total
  FROM '{MARCH}'
""").df()
print(availability.to_string(index=False))
print(f"\nShare available (IS TRUE): {availability['is_true'].iloc[0] / availability['total'].iloc[0] * 100:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 is_true  is_false   is_null   total
413966.0 6408671.0 3018741.0 9841378

Share available (IS TRUE): 4.2%


**Query 3 of 3 — availability, filtered with `IS TRUE` on purpose.** Only 4.2% of March rows
have `ga4_data_available IS TRUE` (413,966 of 9,841,378) — 65.1% are explicitly `IS FALSE`, and
30.7% are `NULL`. This is exactly the trap the assignment card and `flyrank-data` skill warn
about: a plain `= FALSE` or `NOT ga4_data_available` filter would silently mishandle a third of
all rows (the `NULL`s aren't "unavailable," they're *unknown*), and treating every non-TRUE row
as "no engagement, so zero" would be wrong for both the FALSE and NULL rows alike. Any GA4-based
feature (engagement rate, sessions) is only trustworthy on that 4.2%, which matters for the
feature frame below.

In [6]:
# --- The five-feature frame (not one of the three verification queries above,
# a separate deliverable item), at the stated grain (one row = one
# content item on one day), LIMIT for a readable preview -- not aggregated,
# so it stays faithful to the unit of analysis from section 1.
features_preview = con.execute(f"""
  SELECT
    report_date, client_hash_id, content_hash_id,
    gsc_avg_position AS avg_position,
    gsc_impressions AS impressions,
    CASE WHEN gsc_impressions > 0 THEN ROUND(gsc_clicks * 100.0 / gsc_impressions, 3) END AS ctr_pct,
    CASE WHEN ga4_data_available IS TRUE AND ga4_sessions > 0
         THEN ROUND(ga4_engaged_sessions * 100.0 / ga4_sessions, 1) END AS engagement_rate_pct,
    sessions_organic
  FROM '{MARCH}'
  WHERE gsc_impressions > 0
  LIMIT 8
""").df()
print(features_preview.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

report_date          client_hash_id          content_hash_id  avg_position  impressions  ctr_pct  engagement_rate_pct  sessions_organic
 2026-03-01 client_73cda7b4e4f265ea content_b7e512995f79d5a6      3.350000           20    0.000                  NaN              <NA>
 2026-03-01 client_73cda7b4e4f265ea content_05597932fe4da067      0.000000            1    0.000                  NaN              <NA>
 2026-03-01 client_73cda7b4e4f265ea content_7a105f548d9c6916      4.928000          125    0.800                  NaN              <NA>
 2026-03-01 client_73cda7b4e4f265ea content_905aa32a0230694e      4.000000            7    0.000                  NaN              <NA>
 2026-03-01 client_73cda7b4e4f265ea content_a3ea9792f793ec72      2.272727           11    0.000                  NaN              <NA>
 2026-03-01 client_73cda7b4e4f265ea content_36c36abc7650d7af      7.347280          239    0.418                  NaN              <NA>
 2026-03-01 client_73cda7b4e4f265ea content_a7da

**Five features, each "knowable at the decision moment because…":**

1. `avg_position` — that day's own GSC search ranking; observed the same day, nothing later
   involved.
2. `impressions` — that day's own GSC impression count; same reasoning.
3. `ctr_pct` — computed from that same day's `gsc_clicks` / `gsc_impressions`; a same-day ratio,
   not a future one.
4. `engagement_rate_pct` — `ga4_engaged_sessions` / `ga4_sessions`, gated on
   `ga4_data_available IS TRUE`; knowable the same day *when* GA4 data exists for that
   row — which, per Query 2, is only ~4.2% of March rows, so this feature will be `NULL` far
   more often than populated this month.
5. `sessions_organic` — that day's own organic-channel session count from the fact table
   directly, no dimension join needed at all.

All five deliberately avoid `dim_content`'s date columns — see section 4 for why.

In [7]:
# --- THE TRAP: build the label from an observed future window, then
# deliberately add a feature from that same future window and watch the
# "quick score" jump toward perfect -- then remove it. ---
APRIL = f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet"

march_roll = con.execute(f"""
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impr_mar,
         AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_pos_mar
  FROM '{MARCH}' GROUP BY 1, 2
""").df()
april_roll = con.execute(f"""
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impr_apr
  FROM '{APRIL}' GROUP BY 1, 2
""").df()

m = march_roll.merge(april_roll, on=["client_hash_id", "content_hash_id"], how="inner")
m = m[m["impr_mar"] > 0].copy()
m["pct_change"] = (m["impr_apr"] - m["impr_mar"]) / m["impr_mar"] * 100
m["is_declining_proxy"] = (m["pct_change"] < -20).astype(int)
print(f"content items with march impressions > 0: {len(m):,}   declining rate: {m['is_declining_proxy'].mean():.3f}")

from sklearn.metrics import roc_auc_score

honest = m.dropna(subset=["avg_pos_mar"])
auc_honest = roc_auc_score(honest["is_declining_proxy"], -honest["avg_pos_mar"])
print(f"\nHONEST feature (march avg_position, known before April even starts): AUC = {auc_honest:.3f}")

# the deliberate leak: a feature built from the SAME april window used to define the label
m["future_pct_change_feature"] = m["pct_change"]
auc_leaky = roc_auc_score(m["is_declining_proxy"], -m["future_pct_change_feature"])
print(f"LEAKY feature (future pct-change -- the label's own input, smuggled in as a 'feature'): AUC = {auc_leaky:.3f}")

# now delete it and keep only the honest number
m = m.drop(columns=["future_pct_change_feature"])
print(f"\nLeaky column removed. Columns remaining: {list(m.columns)}")
print(f"Honest number to report going forward: AUC = {auc_honest:.3f} (not {auc_leaky:.3f})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content items with march impressions > 0: 176,737   declining rate: 0.532



HONEST feature (march avg_position, known before April even starts): AUC = 0.522
LEAKY feature (future pct-change -- the label's own input, smuggled in as a 'feature'): AUC = 1.000

Leaky column removed. Columns remaining: ['client_hash_id', 'content_hash_id', 'impr_mar', 'avg_pos_mar', 'impr_apr', 'pct_change', 'is_declining_proxy']
Honest number to report going forward: AUC = 0.522 (not 1.000)


The trap performs exactly as advertised. The honest feature — March's own average position,
fully knowable before April even starts — gets AUC 0.522, barely better than a coin flip. The
moment I add `future_pct_change_feature`, which is literally the same April-vs-March ratio the
label is thresholded on, the score jumps to **AUC = 1.000** — not "very good," *perfect*,
because it isn't predicting the label, it's restating it. That's the giveaway: a score that
looks too good to be true, on a problem this genuinely hard, usually is. I deleted the column
immediately after confirming the jump, and the number I'd actually report is 0.522, not 1.000.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# --- how stale is the "days since update" idea, once we know dim_content is a snapshot? ---
future_updates = con.execute(f"""
  SELECT COUNT(*) rows_where_dim_content_update_is_after_report_date, COUNT(*) OVER () AS _dummy
  FROM '{MARCH}' f
  JOIN '{BASE}/dim_content.parquet' c USING (client_hash_id, content_hash_id)
  WHERE c.content_updated_date > f.report_date
""").df()
total_joined = con.execute(f"""
  SELECT COUNT(*) n FROM '{MARCH}' f JOIN '{BASE}/dim_content.parquet' c USING (client_hash_id, content_hash_id)
""").df().iloc[0, 0]
bad = future_updates.iloc[0, 0]
print(f"March rows where dim_content.content_updated_date is AFTER that row's report_date: {bad:,} of {total_joined:,} ({bad/total_joined*100:.1f}%)")

# secondary limitation: how sparse is GA4 coverage this month?
ga4 = con.execute(f"""
  SELECT
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS is_true,
    COUNT(*) AS total
  FROM '{MARCH}'
""").df()
print(f"\nGA4-available rows in March: {ga4['is_true'].iloc[0]:,} of {ga4['total'].iloc[0]:,} ({ga4['is_true'].iloc[0]/ga4['total'].iloc[0]*100:.1f}%)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows where dim_content.content_updated_date is AFTER that row's report_date: 8,672,105 of 9,841,378 (88.1%)



GA4-available rows in March: 413,966.0 of 9,841,378 (4.2%)


### 4. Data limits

**Named limitation: `dim_content`'s date columns describe the export moment, not history.**
88.1% of March rows (8,672,105 of 9,841,378) show `content_updated_date` *after* that row's own
`report_date` — because `dim_content` is a single current-state snapshot taken at export time
(2026-07-03), not a slowly-changing dimension with a value-as-of-each-day. A March row's page may
well have been edited in April, May, or June, and the snapshot only remembers the latest edit.
Computing "days since last update, as of March" from this column would silently use information
from months after the row it's attached to — a subtler, structural version of the same mistake
Section 3's trap makes on purpose. This is why the five-feature frame above uses nothing from
`dim_content` at all: every feature comes straight from that day's own fact-table row.

**Secondary, related limitation:** GA4 coverage this month is thin (4.2% `IS TRUE`, Query 2), so
any engagement-based feature or label built on GA4 signals would need to either accept heavy
missingness in March specifically, or be restricted to the ~4% of rows where it's actually
observed — not imputed as zero. This data can tell me about search-visible pages with GA4 access
this month; it can't tell me much yet about engagement on the other 96%.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.